In [ ]:
%load_ext autoreload
%autoreload 2
import json
from dotenv import load_dotenv
import json
import os

In [ ]:
load_dotenv()

GIRDER_API_URL = os.getenv("GIRDER_API_URL")
API_KEY = os.getenv("HTMDEC_GIRDER_API_KEY")
ALPSS_FORM_ID = '69b2778b7590bb7e1ed158a1'
PDV_FOLDER_ID = '69c3ef40c855e9e387f24425'

In [ ]:
from girder_client import GirderClient
client = GirderClient(apiUrl=GIRDER_API_URL)
client.authenticate(apiKey=API_KEY)
user = client.get("user/me")
print(f"✅ Authenticated as {user['login']}")

In [ ]:
igsns = client.get('deposition', parameters={'limit': 10000,  'sort': 'created', 'sortdir': -1})
# igsns = igsns[:30]
igsns = [{"igsn": 'JHAMAC00003-S1R4C3'}, {"igsn": 'JHAMAC00003-S4R5C3'}, {"igsn": 'JHAMAC00003-S1R6C3'}, {"igsn": 'JHAMAC00001-S8R3C2'}]
# EXLUDE UNNECESSARY IGSN LIKE TAMU
print("number of igsns: ", len(igsns))

In [ ]:
def analyze_files_for_igsn(client, igsn, pdv_items):
    """Single resource/search call for XRD/EBSD + filter pre-fetched PDV items for laser shock."""
    resources = client.get("resource/search", parameters={
        "q": igsn,
        "mode": "igsn",
        "types": '["folder","item"]',
        "limit": 1000,
    })

    xrd_files_count = 0
    ebsd_files_count = 0
    for resource_type_list in resources.values():
        for resource in resource_type_list:
            name = resource.get("name", "")
            if name.endswith("_xrd.csv"):
                xrd_files_count += 1
            elif name.endswith("_ebsd.csv"):
                ebsd_files_count += 1

    shots = [item for item in pdv_items if igsn in item.get("name", "")]
    laser_count = len(shots)

    return {
        "has_xrd": xrd_files_count > 0,
        "xrd_files_count": xrd_files_count,
        "has_ebsd": ebsd_files_count > 0,
        "ebsd_files_count": ebsd_files_count,
        "has_laser_shock": laser_count > 0,
        "laser_shock_shots_count": laser_count,
    }


def analyze_laser_shock_forms_for_igsn(client, igsn):
    """Get quality flag counts (velocity_ok, spall_ok, hel_detected) from ALPSS form entries."""
    results = client.get(
        "entry",
        parameters={
            "formId": ALPSS_FORM_ID,
            "query": igsn,
            "field": "Sample_IGSN",
            "limit": 100000,
        },
    )

    if not results:
        return {
            "velocity_ok_count": 0,
            "spall_ok_count": 0,
            "hel_detected_count": 0,
        }

    return {
        "velocity_ok_count": sum(1 for e in results if e.get("data", {}).get("velocity_ok") is True),
        "spall_ok_count":    sum(1 for e in results if e.get("data", {}).get("spall_ok")    is True),
        "hel_detected_count": sum(1 for e in results if e.get("data", {}).get("hel_detected") is True),
    }


In [ ]:
from tqdm import tqdm

pdv_items = client.get("item", parameters={"folderId": PDV_FOLDER_ID, "limit": 100000})
print(f"Fetched {len(pdv_items)} PDV items")

analysis_results = {}

for dep in tqdm(igsns, desc="Analyzing IGSNs"):
    igsn = dep["igsn"]

    file_info   = analyze_files_for_igsn(client, igsn, pdv_items)
    shock_forms = analyze_laser_shock_forms_for_igsn(client, igsn)

    has_any_data = (
        file_info["has_xrd"]
        or file_info["has_ebsd"]
        or file_info["has_laser_shock"]
    )

    analysis_results[igsn] = {
        "has_any_data": has_any_data,
        **file_info,
        **shock_forms,
    }


In [ ]:
summary = {
    "total_igsns": len(analysis_results),
    "igsns_with_any_data": 0,
    "igsns_with_xrd": 0,
    "igsns_with_ebsd": 0,
    "igsns_with_laser_shock": 0,
    "total_xrd_files": 0,
    "total_ebsd_files": 0,
    "total_laser_shock_entries": 0,
}

for data in analysis_results.values():
    if data["has_any_data"]:
        summary["igsns_with_any_data"] += 1
    if data["has_xrd"]:
        summary["igsns_with_xrd"] += 1
    if data["has_ebsd"]:
        summary["igsns_with_ebsd"] += 1
    if data["has_laser_shock"]:
        summary["igsns_with_laser_shock"] += 1

    summary["total_xrd_files"] += data["xrd_files_count"]
    summary["total_ebsd_files"] += data["ebsd_files_count"]
    summary["total_laser_shock_entries"] += data["laser_shock_shots_count"]

print(json.dumps(summary, indent=3))


In [ ]:
import pandas as pd
df = (
    pd.DataFrame.from_dict(analysis_results, orient="index")
    .reset_index()
    .rename(columns={"index": "igsn"})
)

df = df.sort_values(
    by=["has_xrd", "has_laser_shock"],
    ascending=[False, False],
).reset_index(drop=True)

df.head()

In [ ]:
# 1) Overlap / Venn table
xrd_only     = df[df["has_xrd"] & ~df["has_laser_shock"]]
shock_only   = df[~df["has_xrd"] & df["has_laser_shock"]]
both         = df[df["has_xrd"] & df["has_laser_shock"]]
neither      = df[~df["has_xrd"] & ~df["has_laser_shock"]]
both_spall   = df[df["has_xrd"] & df["has_laser_shock"] & df["spall_ok_count"].gt(0)]
both_hel     = df[df["has_xrd"] & df["has_laser_shock"] & df["hel_detected_count"].gt(0)]
both_spall_hel = df[df["has_xrd"] & df["has_laser_shock"] & df["spall_ok_count"].gt(0) & df["hel_detected_count"].gt(0)]

categories = [
    ("XRD only",                          xrd_only),
    ("Laser shock only",                  shock_only),
    ("Both XRD & laser shock",            both),
    ("Neither",                           neither),
    ("Both XRD & laser shock & spall",    both_spall),
    ("Both XRD & laser shock & HEL",      both_hel),
    ("Both XRD & laser shock & spall & HEL", both_spall_hel),
]

overlap = pd.DataFrame({
    "category": [c for c, _ in categories],
    "count":    [len(d) for _, d in categories],
    "pct":      [round(100*len(d)/len(df), 1) for _, d in categories],
})
print(overlap.to_string(index=False))

In [ ]:
# TODO: location specific analysis

In [ ]:
# 2) Quality flags (velocity_ok, spall_ok, hel_detected) — laser shock IGSNs only
shock_df = df[df["has_laser_shock"]].copy()
total_shock = len(shock_df)

quality = pd.DataFrame({
    "flag":    ["velocity_ok", "spall_ok", "hel_detected"],
    "igsns":   [
        shock_df["velocity_ok_count"].gt(0).sum(),
        shock_df["spall_ok_count"].gt(0).sum(),
        shock_df["hel_detected_count"].gt(0).sum(),
    ],
    "total_shots": [
        shock_df["velocity_ok_count"].sum(),
        shock_df["spall_ok_count"].sum(),
        shock_df["hel_detected_count"].sum(),
    ],
    "total_laser_shots": summary["total_laser_shock_entries"],
})
quality["shot_pct"] = (quality["total_shots"] / quality["total_laser_shots"] * 100).round(1)
print(quality.to_string(index=False))

In [ ]:
# 4) Gap analysis
print("=== XRD but no laser shock ===")
print(xrd_only[["igsn", "xrd_files_count"]].to_string(index=False) if len(xrd_only) else "  (none)")

print("\n=== Laser shock but no XRD ===")
print(shock_only[["igsn", "laser_shock_shots_count"]].to_string(index=False) if len(shock_only) else "  (none)")

In [ ]:
# Dump df to file
df.to_csv("igsn_analysis.csv", index=False)
print("Saved to igsn_analysis.csv")